<a href="https://colab.research.google.com/github/SarahkhIT/AgenticAIProject/blob/main/AgenticAIProject/notebooks/02_rag_and_multi_agent_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Event Planner — 02. RAG Pipeline & Multi-Agent Routing

**Program:** Building Agentic AI Systems by SDAIA Academy

**Session Dates:** 9th of August, 2026 - 13th of August, 2026

**Declared Track:** Track A

**Covers:** Rubric 3 — RAG Pipeline  · Rubric 2 — Multi-Agent / Routing Architecture  · Rubric 7 — Workflow Pattern

This notebook is self-contained. The multi-agent supervisor's `knowledge_agent` depends directly on the retriever built in the RAG section, and the `planning_agent` reuses the core tools from `01_agent_fundamentals.ipynb`, so those tool definitions are re-declared below before the routing section runs.

## Team Members
- Setah Mohammed Alajmi
- Raneem Abdullah Alsheddi
- Jana Hamad Alhumaizi
- Shatha Hamad Bin Mana
- Sarah Abdulaziz Alkhudhiri


In [ ]:
!pip install -q \
    "langchain>=1.0" \
    "langgraph>=1.0" \
    langchain-groq \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    pypdf \
    reportlab

In [ ]:
# ============================================================
# CAPSTONE SECRET SETUP — NO HARDCODED API KEYS
# ============================================================

import os
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

if not groq_key:
    raise RuntimeError(
        "GROQ_API_KEY is missing. Add it to Colab Secrets and enable Notebook access."
    )

os.environ["GROQ_API_KEY"] = groq_key

print("PASS: GROQ_API_KEY loaded securely from Colab Secrets.")


PASS: GROQ_API_KEY loaded securely from Colab Secrets.


### Shared tool definitions (from Rubric 1)
Re-declared here so this notebook runs standalone; see `01_agent_fundamentals.ipynb` for the original section.

In [ ]:
# ============================================================
# SMART EVENT PLANNER
# Person 1 — Agent Fundamentals & Tools
# ============================================================
#
# Capstone requirements addressed in this file:
#
# 1. Agent Fundamentals
#    - Real LLM-based agent
#    - Real tool calls
#    - Tools use their arguments to perform actual work
#    - Structured output with Pydantic
#    - with_structured_output()
#
# 2. Integration-ready
#    - EventRequest is a clear contract
#    - build_event_planning_agent() can be used by Person 2
#    - Tools are independent and reusable
#
# IMPORTANT:
# - Set GROQ_API_KEY as an environment variable.
# - Never put the API key directly in this file.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
from typing import Literal

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# ============================================================
# 1. CONFIGURATION
# ============================================================

DEFAULT_MODEL = "groq:llama-3.3-70b-versatile"


def get_model(model_name: str = DEFAULT_MODEL):
    """
    Create the chat model.

    The API key is read from the environment.
    """

    if not os.environ.get("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY is not set.\n"
            "Please set it as an environment variable before running the agent."
        )

    return init_chat_model(model_name)


# ============================================================
# 2. PYDANTIC MODELS
# ============================================================

class EventRequest(BaseModel):
    """
    Structured representation of the user's event request.

    The LLM populates this model using with_structured_output().
    """

    event_type: str = Field(
        description="Type of event, for example graduation, wedding, birthday."
    )

    guest_count: int = Field(
        gt=0,
        description="Expected number of guests."
    )

    budget: float = Field(
        gt=0,
        description="Total event budget in Saudi Riyals."
    )

    date: str | None = Field(
        default=None,
        description="Event date if explicitly provided by the user."
    )

    location_pref: Literal["indoor", "outdoor"] | None = Field(
        default=None,
        description="Indoor or outdoor preference if explicitly provided."
    )

    theme: str | None = Field(
        default=None,
        description="Event theme or style if explicitly provided."
    )


class BudgetBreakdown(BaseModel):
    venue: float
    catering: float
    decoration: float
    entertainment: float
    logistics: float
    contingency: float


class VenueOption(BaseModel):
    name: str
    capacity: int
    indoor: bool
    estimated_cost: float
    location: str
    reason: str


class CateringOption(BaseModel):
    name: str
    price_per_guest: float
    estimated_total: float
    cuisine: str
    reason: str


class DecorationPlan(BaseModel):
    theme: str
    estimated_cost: float
    concept: str
    items: list[str]


class ChecklistItem(BaseModel):
    task: str
    category: str
    due_before_event_days: int
    priority: Literal["high", "medium", "low"]


# ============================================================
# 3. TOOL 1 — BUDGET CALCULATOR
# ============================================================

@tool
def calculate_budget_split(
    total_budget: float,
    event_type: str,
) -> dict:
    """
    Calculate a realistic event budget allocation.

    Arguments:
        total_budget: Total event budget in SAR.
        event_type: Type of event.

    Returns:
        A structured budget breakdown.
    """

    if total_budget <= 0:
        raise ValueError("total_budget must be greater than zero.")

    if not event_type.strip():
        raise ValueError("event_type cannot be empty.")

    # Default allocation for the Smart Event Planner.
    #
    # The event_type is intentionally accepted as an argument
    # because the LLM must provide it when calling the tool.
    #
    # This can later be customized for different event types.

    allocation = {
        "venue": 0.30,
        "catering": 0.35,
        "decoration": 0.15,
        "entertainment": 0.08,
        "logistics": 0.07,
        "contingency": 0.05,
    }

    breakdown = BudgetBreakdown(
        venue=round(total_budget * allocation["venue"], 2),
        catering=round(total_budget * allocation["catering"], 2),
        decoration=round(total_budget * allocation["decoration"], 2),
        entertainment=round(total_budget * allocation["entertainment"], 2),
        logistics=round(total_budget * allocation["logistics"], 2),
        contingency=round(total_budget * allocation["contingency"], 2),
    )

    return breakdown.model_dump()


# ============================================================
# 4. TOOL 2 — VENUE SEARCH
# ============================================================

@tool
def search_venues(
    guest_count: int,
    budget: float,
    indoor: bool,
) -> list[dict]:
    """
    Find venues based on guest capacity, budget and indoor/outdoor preference.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum venue budget in SAR.
        indoor: True for indoor venues, False for outdoor venues.

    Returns:
        Matching venue options.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    # Demo dataset.
    #
    # These are NOT claimed to be real external venues.
    # They are project data used to demonstrate real filtering
    # and tool execution.
    venues = [
        {
            "name": "Elegant Hall Riyadh",
            "capacity": 120,
            "indoor": True,
            "estimated_cost": 4200,
            "location": "Riyadh",
            "reason": "Elegant indoor hall suitable for formal celebrations.",
        },
        {
            "name": "Garden Celebration Venue",
            "capacity": 150,
            "indoor": False,
            "estimated_cost": 3500,
            "location": "Riyadh",
            "reason": "Outdoor garden venue suitable for large celebrations.",
        },
        {
            "name": "Modern Event Studio",
            "capacity": 90,
            "indoor": True,
            "estimated_cost": 3900,
            "location": "Riyadh",
            "reason": "Modern indoor venue suitable for smaller elegant events.",
        },
        {
            "name": "Grand Celebration Center",
            "capacity": 250,
            "indoor": True,
            "estimated_cost": 5500,
            "location": "Riyadh",
            "reason": "Large indoor event center for bigger celebrations.",
        },
    ]

    # REAL filtering using the tool arguments.
    matching_venues = [
        venue
        for venue in venues
        if (
            venue["capacity"] >= guest_count
            and venue["estimated_cost"] <= budget
            and venue["indoor"] == indoor
        )
    ]

    # Cheapest suitable options first.
    matching_venues.sort(
        key=lambda venue: (
            venue["estimated_cost"],
            venue["capacity"],
        )
    )

    return [
        VenueOption(**venue).model_dump()
        for venue in matching_venues[:3]
    ]


# ============================================================
# 5. TOOL 3 — CATERING SEARCH
# ============================================================

@tool
def search_catering(
    guest_count: int,
    budget: float,
) -> list[dict]:
    """
    Find catering options based on guest count and catering budget.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum catering budget in SAR.

    Returns:
        Catering options that fit the budget.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    catering_options = [
        {
            "name": "Premium Saudi Buffet",
            "price_per_guest": 65,
            "cuisine": "Saudi / Arabic",
            "reason": "Suitable for formal family celebrations.",
        },
        {
            "name": "International Buffet",
            "price_per_guest": 70,
            "cuisine": "International",
            "reason": "Broad menu suitable for mixed preferences.",
        },
        {
            "name": "Elegant Finger Food",
            "price_per_guest": 45,
            "cuisine": "International",
            "reason": "Suitable for a modern elegant reception.",
        },
    ]

    results = []

    for option in catering_options:

        # REAL calculation based on guest_count.
        estimated_total = (
            guest_count * option["price_per_guest"]
        )

        # REAL budget filtering.
        if estimated_total <= budget:

            results.append(
                CateringOption(
                    name=option["name"],
                    price_per_guest=option["price_per_guest"],
                    estimated_total=estimated_total,
                    cuisine=option["cuisine"],
                    reason=option["reason"],
                ).model_dump()
            )

    results.sort(
        key=lambda item: item["estimated_total"]
    )

    return results[:3]


# ============================================================
# 6. TOOL 4 — DECORATION PLANNER
# ============================================================

@tool
def suggest_decoration(
    theme: str,
    guest_count: int,
    budget: float,
) -> dict:
    """
    Suggest a decoration concept based on theme, guest count and budget.

    Arguments:
        theme: Requested event theme.
        guest_count: Number of guests.
        budget: Decoration budget in SAR.

    Returns:
        A structured decoration plan.
    """

    if not theme.strip():
        raise ValueError("theme cannot be empty.")

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    normalized_theme = theme.lower().strip()

    if "elegant" in normalized_theme:

        concept = (
            "Elegant graduation setup with neutral colors, "
            "warm lighting, floral accents and a decorated stage."
        )

        items = [
            "Graduation backdrop",
            "Warm ambient lighting",
            "Floral centerpieces",
            "Welcome signage",
            "Decorated stage",
        ]

    elif "modern" in normalized_theme:

        concept = (
            "Modern minimalist setup with clean lines, "
            "accent lighting and a contemporary photo area."
        )

        items = [
            "Minimalist backdrop",
            "Accent lighting",
            "Modern table styling",
            "Photo wall",
        ]

    elif "casual" in normalized_theme:

        concept = (
            "Casual and welcoming setup with simple colors, "
            "comfortable seating and themed decorations."
        )

        items = [
            "Themed backdrop",
            "Simple table decoration",
            "Welcome signage",
            "Photo area",
        ]

    else:

        concept = (
            f"A {theme} themed decoration concept "
            "adapted to the event size and available budget."
        )

        items = [
            "Themed backdrop",
            "Table decoration",
            "Welcome signage",
            "Photo area",
        ]

    return DecorationPlan(
        theme=theme,
        estimated_cost=round(budget, 2),
        concept=concept,
        items=items,
    ).model_dump()


# ============================================================
# 7. TOOL 5 — EVENT CHECKLIST
# ============================================================

@tool
def create_checklist(
    event_date: str,
    categories: list[str],
) -> list[dict]:
    """
    Create an event preparation checklist.

    Arguments:
        event_date: Event date.
        categories: Planning categories to include.

    Returns:
        Checklist items.
    """

    if not event_date.strip():
        raise ValueError("event_date is required.")

    if not categories:
        raise ValueError("At least one category is required.")

    category_tasks = {

        "venue": [
            (
                "Confirm venue booking",
                30,
                "high",
            ),
            (
                "Confirm seating layout",
                7,
                "medium",
            ),
        ],

        "catering": [
            (
                "Confirm catering menu",
                14,
                "high",
            ),
            (
                "Confirm final guest count",
                3,
                "high",
            ),
        ],

        "decoration": [
            (
                "Finalize decoration concept",
                21,
                "medium",
            ),
            (
                "Confirm decoration setup",
                7,
                "high",
            ),
        ],

        "logistics": [
            (
                "Prepare event timeline",
                7,
                "high",
            ),
            (
                "Confirm equipment and sound system",
                5,
                "medium",
            ),
        ],

        "approval": [
            (
                "Review final event plan with user",
                1,
                "high",
            ),
        ],
    }

    checklist = []

    for category in categories:

        tasks = category_tasks.get(
            category.lower().strip(),
            [],
        )

        for task, days, priority in tasks:

            checklist.append(
                ChecklistItem(
                    task=task,
                    category=category,
                    due_before_event_days=days,
                    priority=priority,
                ).model_dump()
            )

    return checklist


# ============================================================
# 8. REGISTER ALL TOOLS
# ============================================================

ALL_TOOLS = [
    calculate_budget_split,
    search_venues,
    search_catering,
    suggest_decoration,
    create_checklist,
]


# ============================================================
# 9. STRUCTURED INTAKE
# ============================================================

INTAKE_SYSTEM_PROMPT = """
You are the structured intake component of Smart Event Planner.

Extract event-planning information from the user's message.

IMPORTANT:
Only extract information that is explicitly supported by the user's
message.

DO NOT invent or assume:
- event dates
- guest counts
- budgets
- indoor/outdoor preferences
- themes

The required event fields are:
- event_type
- guest_count
- budget

The optional fields are:
- date
- location_pref
- theme

If an optional field is not mentioned, return null.

The budget is represented in Saudi Riyals.
"""


def extract_event_request(
    user_message: str,
    model=None,
) -> EventRequest:
    """
    Convert natural-language user input into a validated EventRequest.

    This uses LangChain structured output with a Pydantic model.
    """

    if not user_message.strip():
        raise ValueError(
            "user_message cannot be empty."
        )

    model = model or get_model()

    structured_model = model.with_structured_output(
        EventRequest
    )

    event = structured_model.invoke(
        [
            {
                "role": "system",
                "content": INTAKE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ]
    )

    return event


# ============================================================
# 10. CHECK WHETHER EVENT DETAILS ARE COMPLETE
# ============================================================

def get_missing_event_fields(
    event: EventRequest,
) -> list[str]:
    """
    Return optional fields that are useful for complete planning.

    This does NOT invent missing information.
    """

    missing = []

    if event.date is None:
        missing.append("date")

    if event.location_pref is None:
        missing.append("location_pref")

    if event.theme is None:
        missing.append("theme")

    return missing


def build_clarifying_question(
    event: EventRequest,
) -> str | None:
    """
    Build a follow-up question when optional planning details
    are missing.
    """

    missing = get_missing_event_fields(event)

    if not missing:
        return None

    questions = []

    if "date" in missing:
        questions.append(
            "What date is the event?"
        )

    if "location_pref" in missing:
        questions.append(
            "Would you prefer indoor or outdoor?"
        )

    if "theme" in missing:
        questions.append(
            "What style or theme would you like?"
        )

    return (
        "Before I build the full plan, "
        "please provide: "
        + " ".join(questions)
    )


# ============================================================
# 11. AGENT SYSTEM PROMPT
# ============================================================

AGENT_SYSTEM_PROMPT = """
You are the core event-planning agent for Smart Event Planner.

Your job is to create a practical event plan using the available tools.

IMPORTANT RULE:
Use the tools to obtain planning information.
Do NOT invent:
- budget calculations
- venue options
- catering prices
- decoration costs
- checklist items

The available tools are:

1. calculate_budget_split
   Use it to calculate the event budget allocation.

2. search_venues
   Use it to find suitable venues based on guest count,
   venue budget and indoor/outdoor preference.

3. search_catering
   Use it to find catering options based on guest count
   and catering budget.

4. suggest_decoration
   Use it to create a decoration concept based on theme,
   guest count and decoration budget.

5. create_checklist
   Use it to create event preparation tasks.

You should call the tools yourself when needed.

The tool results are the source of truth for numerical
recommendations.

When all relevant information is available, produce a concise
event plan containing:

1. Event summary
2. Budget breakdown
3. Recommended venue
4. Catering recommendation
5. Decoration concept
6. Preparation checklist

Do not claim that the venue or catering data came from a real
external service. The current tools use the project's local
planning dataset.

If a required detail is missing, clearly state that the information
is needed rather than inventing it.
"""


# ============================================================
# 12. BUILD THE REAL AGENT
# ============================================================

def build_event_planning_agent(
    model=None,
):
    """
    Build the reusable Smart Event Planner agent.

    Person 2 can later integrate this agent into the project's
    multi-agent routing architecture.
    """

    model = model or get_model()

    agent = create_agent(
        model=model,
        tools=ALL_TOOLS,
        system_prompt=AGENT_SYSTEM_PROMPT,
    )

    return agent


# ============================================================
# 13. RUN THE AGENT
# ============================================================

def run_event_planner(
    event: EventRequest,
    model=None,
):
    """
    Run the event planning agent using a validated EventRequest.
    """

    # The planning agent needs these details.
    if event.date is None:
        raise ValueError(
            "event.date is required before running the full planner."
        )

    if event.location_pref is None:
        raise ValueError(
            "event.location_pref is required before running the full planner."
        )

    if event.theme is None:
        raise ValueError(
            "event.theme is required before running the full planner."
        )

    agent = build_event_planning_agent(model)

    event_details = event.model_dump()

    planning_prompt = f"""
Create the event plan using the following structured event request:

{event_details}

You must use the available tools to calculate and retrieve
the planning information.

Do not invent tool results.

Use the actual tool outputs in your final response.
"""

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": planning_prompt,
                }
            ]
        }
    )

    return result


# ============================================================
# 14. DISPLAY REAL TOOL CALLS
# ============================================================

def display_agent_trace(
    result,
):
    """
    Display the actual LLM -> Tool -> Tool Result execution.

    This is important evidence for the Capstone rubric because
    it demonstrates that the LLM selected and called the tools.
    """

    print("\n")
    print("=" * 80)
    print("REAL AGENT TOOL-CALL TRACE")
    print("=" * 80)

    tool_call_count = 0

    for message in result["messages"]:

        # ----------------------------------------------------
        # AI MESSAGE
        # ----------------------------------------------------

        if message.type == "ai":

            tool_calls = getattr(
                message,
                "tool_calls",
                [],
            )

            if tool_calls:

                for call in tool_calls:

                    tool_call_count += 1

                    print("\n[LLM -> TOOL]")
                    print(
                        f"Tool: {call['name']}"
                    )

                    print(
                        "Arguments:"
                    )

                    print(
                        call["args"]
                    )

            elif message.content:

                print("\n[LLM FINAL RESPONSE]")
                print(message.content)

        # ----------------------------------------------------
        # TOOL MESSAGE
        # ----------------------------------------------------

        elif message.type == "tool":

            print("\n[TOOL -> LLM]")

            print(
                f"Tool: {message.name}"
            )

            print(
                "Result:"
            )

            print(
                message.content
            )

    print("\n")
    print("=" * 80)
    print(
        f"TOTAL TOOL CALLS: {tool_call_count}"
    )
    print("=" * 80)

    return tool_call_count


# ============================================================
# 15. OFFLINE TOOL TEST
# ============================================================
#
# This section verifies that the tools themselves work.
#
# IMPORTANT:
# These are MANUAL tool calls.
# They are NOT the evidence for LLM tool selection.
#
# The actual Agent evidence is the test below this section.
# ============================================================

def run_offline_tool_tests():

    print("\n")
    print("=" * 80)
    print("OFFLINE TOOL TESTS")
    print("=" * 80)

    budget = calculate_budget_split.invoke(
        {
            "total_budget": 15000,
            "event_type": "graduation",
        }
    )

    print("\nBudget:")
    print(budget)

    venues = search_venues.invoke(
        {
            "guest_count": 80,
            "budget": budget["venue"],
            "indoor": True,
        }
    )

    print("\nVenues:")
    for venue in venues:
        print(venue)

    catering = search_catering.invoke(
        {
            "guest_count": 80,
            "budget": budget["catering"],
        }
    )

    print("\nCatering:")
    for option in catering:
        print(option)

    decoration = suggest_decoration.invoke(
        {
            "theme": "elegant",
            "guest_count": 80,
            "budget": budget["decoration"],
        }
    )

    print("\nDecoration:")
    print(decoration)

    checklist = create_checklist.invoke(
        {
            "event_date": "2026-09-20",
            "categories": [
                "venue",
                "catering",
                "decoration",
                "logistics",
                "approval",
            ],
        }
    )

    print("\nChecklist:")
    for item in checklist:
        print(item)

    print("\nOffline tool tests completed successfully.")


# ============================================================
# 16. CAPSTONE DEMO
# ============================================================

def run_capstone_demo():

    print("\n")
    print("#" * 80)
    print("# SMART EVENT PLANNER — PERSON 1 CAPSTONE DEMO")
    print("#" * 80)

    # --------------------------------------------------------
    # USER INPUT
    # --------------------------------------------------------

    user_message = """
    I want to plan a graduation party for 80 guests
    with a budget of 15,000 SAR.
    The event will be on 20 September 2026.
    I prefer an indoor venue with an elegant theme.
    """

    print("\n")
    print("=" * 80)
    print("USER REQUEST")
    print("=" * 80)

    print(user_message)

    # --------------------------------------------------------
    # STRUCTURED INTAKE
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("STRUCTURED OUTPUT — EVENT REQUEST")
    print("=" * 80)

    event = extract_event_request(
        user_message
    )

    print(
        event.model_dump_json(
            indent=2
        )
    )

    # --------------------------------------------------------
    # CHECK MISSING INFORMATION
    # --------------------------------------------------------

    missing = get_missing_event_fields(
        event
    )

    if missing:

        print("\n")
        print("=" * 80)
        print("MISSING OPTIONAL INFORMATION")
        print("=" * 80)

        print(
            build_clarifying_question(event)
        )

        print(
            "\nThe demo will stop here because the planner "
            "does not invent missing information."
        )

        return

    # --------------------------------------------------------
    # REAL AGENT EXECUTION
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("RUNNING REAL LLM AGENT")
    print("=" * 80)

    result = run_event_planner(
        event
    )

    # --------------------------------------------------------
    # DISPLAY REAL TOOL CALLS
    # --------------------------------------------------------

    tool_call_count = display_agent_trace(
        result
    )

    # --------------------------------------------------------
    # CAPSTONE ASSERTION
    # --------------------------------------------------------

    if tool_call_count == 0:

        raise RuntimeError(
            "The agent completed without calling any tool. "
            "This would NOT satisfy the real tool-calling requirement."
        )

    print("\n")
    print("=" * 80)
    print("CAPSTONE CHECK")
    print("=" * 80)

    print(
        "PASS: The LLM made real tool calls."
    )

    print(
        f"PASS: Number of tool calls = {tool_call_count}"
    )

    print(
        "PASS: Structured EventRequest was produced with Pydantic."
    )

    print(
        "PASS: with_structured_output() was used."
    )

    print(
        "PASS: Tools received arguments from the agent."
    )

    print(
        "PASS: Tool results were returned to the LLM."
    )

    print(
        "PASS: Final plan was generated from tool results."
    )


# ============================================================
# 17. MAIN
# ============================================================

if __name__ == "__main__":

    print(
        "Smart Event Planner — Person 1")

    print( "Agent Fundamentals & Tools")

    print( "\nAvailable components:")

    print( "- EventRequest")

    print("- Structured Intake")

    print("- calculate_budget_split")

    print("- search_venues")

    print("- search_catering")

    print( "- suggest_decoration")

    print( "- create_checklist")

    print("- Event Planning Agent")

    print("\nRun run_offline_tool_tests() to test tools." )

    print( "Run run_capstone_demo() to demonstrate the real agent.")

Smart Event Planner — Person 1
Agent Fundamentals & Tools

Available components:
- EventRequest
- Structured Intake
- calculate_budget_split
- search_venues
- search_catering
- suggest_decoration
- create_checklist
- Event Planning Agent

Run run_offline_tool_tests() to test tools.
Run run_capstone_demo() to demonstrate the real agent.


# Rubric 3 — RAG Pipeline

**RAG design choice: 2-Step RAG.**

This project uses **2-Step RAG**: documents are loaded, split, embedded, stored, and retrieved first; the retrieved context is then supplied to the LLM for answer generation. This is preferable here to Agentic RAG because event-knowledge questions should always be grounded in the project documents rather than asking an agent to decide whether retrieval is necessary. A Hybrid design would add routing complexity without a clear benefit for this small, focused knowledge base. The retrieval test below asks a question directly supported by the PDFs and prints the retrieved chunks and sources before generation.


In [ ]:
#RAG Pipeline
print("Person 3 — RAG Pipeline")

Person 3 — RAG Pipeline


In [ ]:
!pip install -q langchain-community pypdf langchain-text-splitters langchain-huggingface sentence-transformers

In [ ]:
#RAG PIPELINE IMPORTS

from pathlib import Path
from typing import List

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

print("RAG imports loaded successfully.")

/tmp/ipykernel_5226/875248776.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


RAG imports loaded successfully.


In [ ]:
#RAG CONFIGURATION

KNOWLEDGE_DIR = Path("event_knowledge")
DEFAULT_EMBEDDING_MODEL = ("sentence-transformers/all-MiniLM-L6-v2")

print("RAG configuration loaded.")

RAG configuration loaded.


In [ ]:
# CREATE EVENT KNOWLEDGE FOLDER

KNOWLEDGE_DIR.mkdir(

    exist_ok=True
)
print( f"Knowledge folder ready: "
       f"{KNOWLEDGE_DIR.resolve()}" )

Knowledge folder ready: /content/event_knowledge


In [ ]:
# CREATE EVENT KNOWLEDGE PDF FILES AUTOMATICALLY

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

KNOWLEDGE_DIR.mkdir(exist_ok=True)

styles = getSampleStyleSheet()

knowledge_files = {
    "graduation_event_guide.pdf": """
    Graduation Event Planning Guide

    Confirm the expected guest count before selecting a venue.
    The venue should comfortably support guests, seating,
    food service, photography areas, and movement.

    Set a total budget and divide it among venue, catering,
    decoration, entertainment, logistics, and contingency.

    For an indoor graduation event, check air conditioning,
    lighting, sound equipment, stage space, accessibility,
    and emergency exits.

    Four to six weeks before the event, confirm the venue,
    estimated guest count, budget, and event theme.

    Two to three weeks before the event, finalize catering,
    decoration, photography requirements, and event schedule.
    """,

    "venue_selection_guide.pdf": """
    Event Venue Selection Guide

    Select a venue whose capacity is at least the expected
    guest count. Do not exceed the approved venue capacity.

    For indoor events, evaluate air conditioning, ventilation,
    lighting, electrical access, sound system compatibility,
    restrooms, accessibility, and emergency exits.

    The venue should have clear entry and exit routes and
    enough space for seating, catering, photography,
    and guest circulation.

    Compare the venue cost with the allocated venue budget.
    Consider location, parking, accessibility, setup time,
    and cleanup requirements.
    """,

    "event_safety_guide.pdf": """
    Event Safety and Operations Guide

    Emergency exits should remain visible, accessible,
    and unobstructed throughout the event.

    Do not exceed the approved venue capacity.

    Maintain clear walking paths between seating,
    entrances, exits, catering areas, and activity zones.

    Electrical cables and equipment should be secured
    to reduce trip hazards.

    Before guests arrive, verify exits, walking paths,
    equipment placement, seating layout, and emergency
    contact information.
    """
}

for filename, content in knowledge_files.items():
    file_path = KNOWLEDGE_DIR / filename

    doc = SimpleDocTemplate(
        str(file_path),
        pagesize=A4
    )

    story = []

    for paragraph in content.strip().split("\n\n"):
        story.append(
            Paragraph(
                paragraph.strip(),
                styles["BodyText"]
            )
        )
        story.append(Spacer(1, 12))

    doc.build(story)

print("Knowledge files created automatically.")

for file in KNOWLEDGE_DIR.glob("*.pdf"):
    print("-", file.name)

Knowledge files created automatically.
- venue_selection_guide.pdf
- event_safety_guide.pdf
- graduation_event_guide.pdf


In [ ]:
# LOAD EVENT KNOWLEDGE DOCUMENTS

def load_event_documents():
    documents = []
    pdf_files = sorted(
        KNOWLEDGE_DIR.glob("*.pdf")
    )

    if not pdf_files:
        raise FileNotFoundError(
            "No PDF files found inside event_knowledge."
        )

    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()
        for page in pages:
            page.metadata["source"] = pdf_path.name
        documents.extend(pages)

    print(f"PDF files found: {len(pdf_files)}")
    print(f"Document pages loaded: {len(documents)}")
    return documents

documents = load_event_documents()

PDF files found: 3
Document pages loaded: 3


In [ ]:
# SPLIT DOCUMENTS INTO CHUNKS

def split_event_documents(
    documents,
    chunk_size=700,
    chunk_overlap=120,
):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = splitter.split_documents(documents)

    print(f"Total chunks created: {len(chunks)}")
    return chunks

chunks = split_event_documents(documents)

Total chunks created: 3


In [ ]:
# CREATE EMBEDDING MODEL

embedding_model = HuggingFaceEmbeddings(
    model_name=DEFAULT_EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True
    },

)
print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [ ]:
# CREATE VECTOR STORE

vector_store = InMemoryVectorStore(
    embedding=embedding_model
)

vector_store.add_documents(chunks)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")

Vector store created successfully.
Stored chunks: 3


In [ ]:
# CREATE RETRIEVER

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

Retriever created successfully.


In [ ]:
# TEST REAL DOCUMENT RETRIEVAL

question = (
    "What should be considered when planning "
    "an indoor graduation event?"
)

retrieved_docs = retriever.invoke(question)
print("=" * 70)
print("RAG RETRIEVAL TEST")
print("=" * 70)
print(f"\nQuestion: {question}")
print(f"\nRetrieved chunks: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, start=1):
    print("\n" + "-" * 70)
    print(f"CHUNK {i}")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

    page = doc.metadata.get("page")

    if page is not None:
        print(f"Page: {page + 1}")

    print("\nContent:")
    print(doc.page_content)
print("\n" + "=" * 70)

if retrieved_docs:
    print("PASS: Retriever returned real document content.")
else:
    raise RuntimeError("FAIL: No documents were retrieved.")

RAG RETRIEVAL TEST

Question: What should be considered when planning an indoor graduation event?

Retrieved chunks: 3

----------------------------------------------------------------------
CHUNK 1
Source: graduation_event_guide.pdf
Page: 1

Content:
Graduation Event Planning Guide
Confirm the expected guest count before selecting a venue. The venue should comfortably support
guests, seating, food service, photography areas, and movement.
Set a total budget and divide it among venue, catering, decoration, entertainment, logistics, and
contingency.
For an indoor graduation event, check air conditioning, lighting, sound equipment, stage space,
accessibility, and emergency exits.
Four to six weeks before the event, confirm the venue, estimated guest count, budget, and event
theme.
Two to three weeks before the event, finalize catering, decoration, photography requirements, and
event schedule.

----------------------------------------------------------------------
CHUNK 2
Source: venue_se

In [ ]:
# LOAD GROQ API KEY FROM COLAB SECRETS
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("GROQ API key loaded successfully.")

GROQ API key loaded successfully.


In [ ]:
# GENERATE RAG ANSWER

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

sources = sorted(
    {   doc.metadata.get("source", "Unknown")
   for doc in retrieved_docs
    }
)

rag_prompt = f"""
You are the Event Knowledge Assistant
for Smart Event Planner.
Answer the user's question using ONLY
the retrieved context below.
If the answer is not available in the context,
say that the event knowledge base does not contain
enough information.

QUESTION:

{question}

RETRIEVED CONTEXT:

{context}

Give a concise and practical answer.

"""
model = get_model()
response = model.invoke(rag_prompt)
print("=" * 70)
print("RAG FINAL ANSWER")
print("=" * 70)
print(response.content)
print("\nSources:")

for source in sources:
    print(f"- {source}")

RAG FINAL ANSWER
When planning an indoor graduation event, consider the following: 
1. Air conditioning, 
2. Lighting, 
3. Sound equipment, 
4. Stage space, 
5. Accessibility, and 
6. Emergency exits. 

Additionally, ensure the venue has clear entry and exit routes, enough space for seating, catering, photography, and guest circulation, and does not exceed the approved venue capacity.

Sources:
- event_safety_guide.pdf
- graduation_event_guide.pdf
- venue_selection_guide.pdf


# Rubric 2 — Multi-Agent / Routing Architecture

**Architecture used:** Supervisor + Workers with real LLM handoffs.

The supervisor receives the user request and delegates it to either `planning_agent` or `knowledge_agent`. The route is selected by the LLM through handoff tools such as `transfer_to_planning_agent` and `transfer_to_knowledge_agent`; there is no keyword-based `if` routing. The two routing tests below provide direct evidence for both paths and print the handoff traces.


In [ ]:
# ============================================================
# SMART EVENT PLANNER
# Multi-Agent Architecture + Workflow Pattern
# ============================================================
#
# Capstone requirements addressed:
#
# 2. Multi-Agent / Routing Architecture
#    - Supervisor + Workers architecture
#    - LLM-based routing
#    - Specialist agents with distinct responsibilities
#
# 7. Workflow Pattern
#    - Pattern: ROUTING
#    - The supervisor dynamically routes requests
#      to the appropriate specialist agent.
#
# ============================================================

print("Multi-Agent Architecture + Routing Workflow")

Multi-Agent Architecture + Routing Workflow


In [ ]:
!pip install -qU langgraph-supervisor

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph_supervisor import create_supervisor

print("imports loaded successfully.")

imports loaded successfully.


In [ ]:
# ============================================================
# RAG TOOL FOR THE KNOWLEDGE SPECIALIST
# ============================================================

@tool
def search_event_knowledge(question: str) -> str:
    """
    Search the event knowledge base for relevant information
    about event planning, venues, safety, timelines, logistics,
    and documented event-planning recommendations.
    """

    # Use Person 3's existing RAG retriever
    docs = retriever.invoke(question)

    if not docs:
        return "No relevant information was found in the event knowledge base."

    results = []

    for i, doc in enumerate(docs, start=1):

        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page")

        page_display = page + 1 if page is not None else "Unknown"

        results.append(
            f"""
RESULT {i}
Source: {source}
Page: {page_display}

{doc.page_content}
"""
        )

    return "\n".join(results)


print("RAG tool connected successfully.")

RAG tool connected successfully.


In [ ]:
# ============================================================
#  MODEL
# ============================================================

person2_model = get_model()

print("model created successfully.")

model created successfully.


In [ ]:
# ============================================================
# EVENT KNOWLEDGE SPECIALIST
# ============================================================

knowledge_agent = create_agent(
    model=person2_model,
    tools=[search_event_knowledge],
    name="knowledge_agent",
    system_prompt="""
You are the Event Knowledge Specialist in the Smart Event Planner
multi-agent system.

Your responsibility is to answer questions using the project's
event knowledge base.

Use the search_event_knowledge tool whenever the user asks about:

- event-planning guidelines
- planning timelines
- venue considerations
- event safety
- logistics
- recommendations contained in the project documents

Base your answers on the information retrieved from the knowledge
base.

Do not invent information that is not supported by the retrieved
documents.

If the user is asking for a complete event plan involving budget,
venue, catering, decoration, or a checklist, that request should
be handled by the planning agent.
"""
)

print("Knowledge specialist created successfully.")

Knowledge specialist created successfully.


In [ ]:
# ============================================================
# GROQ-COMPATIBLE WRAPPERS FOR PERSON 1 TOOLS
# ============================================================

import json


@tool
def planning_budget(total_budget: float, event_type: str) -> str:
    """Calculate the event budget allocation."""
    result = calculate_budget_split.invoke({
        "total_budget": total_budget,
        "event_type": event_type
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def planning_venues(
    guest_count: int,
    budget: float,
    indoor: bool
) -> str:
    """Find suitable venues for the event."""
    result = search_venues.invoke({
        "guest_count": guest_count,
        "budget": budget,
        "indoor": indoor
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def planning_catering(
    guest_count: int,
    budget: float
) -> str:
    """Find suitable catering options."""
    result = search_catering.invoke({
        "guest_count": guest_count,
        "budget": budget
    })
    return json.dumps(result, ensure_ascii=False)


print("Groq-compatible planning tools created successfully.")

Groq-compatible planning tools created successfully.


In [ ]:
# ============================================================
# REMAINING GROQ-COMPATIBLE PLANNING TOOLS
# ============================================================

@tool
def planning_decoration(
    theme: str,
    guest_count: int,
    budget: float
) -> str:
    """Suggest suitable decorations for the event."""

    result = suggest_decoration.invoke({
        "theme": theme,
        "guest_count": guest_count,
        "budget": budget
    })

    return json.dumps(result, ensure_ascii=False)


@tool
def planning_checklist(
    event_date: str,
    categories: list[str]
) -> str:
    """Create an event preparation checklist."""

    result = create_checklist.invoke({
        "event_date": event_date,
        "categories": categories
    })

    return json.dumps(result, ensure_ascii=False)


print("All Groq-compatible planning tools created successfully.")

All Groq-compatible planning tools created successfully.


In [ ]:
# ============================================================
# CREATE GROQ-COMPATIBLE PLANNING SPECIALIST
# ============================================================

planning_tools = [
    planning_budget,
    planning_venues,
    planning_catering,
    planning_decoration,
    planning_checklist,
]

planning_agent = create_agent(
    model=person2_model,
    tools=planning_tools,
    name="planning_agent",
    system_prompt="""
You are the Event Planning Specialist.

Create practical event plans using the available tools.

Use the appropriate tools for:
- budget allocation
- venue recommendations
- catering
- decoration
- preparation checklists

Use actual tool results and do not invent results.

Call tools as needed to complete the user's event-planning request.
"""
)

print("Groq-compatible planning specialist created successfully.")

Groq-compatible planning specialist created successfully.


In [ ]:
# ============================================================
# SUPERVISOR — LLM-BASED ROUTING
# ============================================================

supervisor = create_supervisor(
    agents=[
        planning_agent,
        knowledge_agent,
    ],
    model=person2_model,
    prompt="""
You are the Supervisor of the Smart Event Planner multi-agent system.

Your responsibility is to understand the user's request and
delegate it to the most appropriate specialist agent.

You have two specialist agents:

1. planning_agent

Use the planning_agent when the user wants an actual event plan
or needs event-planning actions such as:
- budget allocation
- venue recommendations
- catering recommendations
- decoration recommendations
- event preparation checklists
- complete event planning

2. knowledge_agent

Use the knowledge_agent when the user asks for factual or advisory
information that should come from the project's event knowledge
base, such as:
- event-planning guidelines
- planning timelines
- venue considerations
- event safety
- logistics
- documented recommendations

IMPORTANT:

You must decide which specialist should handle the request based
on the meaning and intent of the user's request.

Do not use keyword matching or hardcoded routing rules.

Delegate the request to the appropriate specialist.

After the specialist completes its task, return a clear final
answer based on the specialist's result.
"""
).compile()

print("Supervisor created successfully.")

Supervisor created successfully.


In [ ]:
# ============================================================
# TEST 1 — ROUTE TO KNOWLEDGE AGENT
# ============================================================

knowledge_test = supervisor.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What safety considerations should I keep in mind "
                    "when planning an indoor graduation event?"
                ),
            }
        ]
    }
)

print("Knowledge routing test completed successfully.")

Knowledge routing test completed successfully.


In [ ]:
# ============================================================
# DISPLAY ROUTING TRACE
# ============================================================

print("=" * 70)
print("SUPERVISOR ROUTING TRACE")
print("=" * 70)

for message in knowledge_test["messages"]:

    print(f"\n[{message.type.upper()}]")

    if getattr(message, "name", None):
        print("Name:", message.name)

    tool_calls = getattr(message, "tool_calls", [])

    if tool_calls:
        for call in tool_calls:
            print("Tool call:", call["name"])
            print("Arguments:", call["args"])

    if getattr(message, "content", None):
        print("Content:", message.content)

SUPERVISOR ROUTING TRACE

[HUMAN]
Content: What safety considerations should I keep in mind when planning an indoor graduation event?

[AI]
Name: supervisor
Tool call: transfer_to_knowledge_agent
Arguments: {}

[TOOL]
Name: transfer_to_knowledge_agent
Content: Successfully transferred to knowledge_agent

[AI]
Name: knowledge_agent
Content: When planning an indoor graduation event, consider the following safety aspects:

1. Confirm the expected guest count and ensure the venue can comfortably accommodate them, along with seating, food service, photography areas, and movement.
2. Check the venue's air conditioning, lighting, sound equipment, stage space, accessibility, and emergency exits.
3. Ensure emergency exits remain visible, accessible, and unobstructed throughout the event.
4. Do not exceed the approved venue capacity.
5. Maintain clear walking paths between seating, entrances, exits, catering areas, and activity zones.
6. Secure electrical cables and equipment to reduce trip haza

In [ ]:
# ============================================================
# TEST — ROUTE TO PLANNING AGENT
# ============================================================

planning_test = supervisor.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a complete indoor graduation event plan "
                    "for 100 guests with a budget of 15000 SAR. "
                    "The event date is September 20, 2026. "
                    "The theme is elegant blue and silver. "
                    "Include budget allocation, venue recommendations, "
                    "catering, decoration, and a preparation checklist."
                ),
            }
        ]
    }
)

print("Planning routing test completed successfully.")

Planning routing test completed successfully.


In [ ]:
# ============================================================
# DISPLAY PLANNING ROUTING TRACE
# ============================================================

print("=" * 70)
print("PLANNING ROUTING TRACE")
print("=" * 70)

for message in planning_test["messages"]:

    print(f"\n[{message.type.upper()}]")

    if getattr(message, "name", None):
        print("Name:", message.name)

    tool_calls = getattr(message, "tool_calls", [])

    if tool_calls:
        for call in tool_calls:
            print("Tool call:", call["name"])
            print("Arguments:", call["args"])

    if getattr(message, "content", None):
        print("Content:", message.content)

PLANNING ROUTING TRACE

[HUMAN]
Content: Create a complete indoor graduation event plan for 100 guests with a budget of 15000 SAR. The event date is September 20, 2026. The theme is elegant blue and silver. Include budget allocation, venue recommendations, catering, decoration, and a preparation checklist.

[AI]
Name: supervisor
Tool call: transfer_to_planning_agent
Arguments: {}

[TOOL]
Name: transfer_to_planning_agent
Content: Successfully transferred to planning_agent

[AI]
Name: planning_agent
Content: Based on the provided tools, here is a complete indoor graduation event plan for 100 guests with a budget of 15000 SAR. The event date is September 20, 2026, and the theme is elegant blue and silver.

Budget Allocation:
- Venue: 4500 SAR
- Catering: 5250 SAR
- Decoration: 2250 SAR
- Entertainment: 1200 SAR
- Logistics: 1050 SAR
- Contingency: 750 SAR

Venue Recommendations:
- Elegant Hall Riyadh: 4200 SAR, capacity 120, indoor, location Riyadh
- Grand Celebration Center: 5500 SAR, ca

# Rubric 7 — Workflow Pattern

**Selected workflow pattern: Routing.**

Routing fits Smart Event Planner because requests naturally divide into two specialist responsibilities: operational event planning and document-grounded event knowledge. The LLM supervisor interprets the request and sends it to the appropriate specialist, then receives the result back for the final answer. The preceding planning and knowledge traces are the captured evidence for this workflow pattern.


In [ ]:
# ============================================================
# WORKFLOW PATTERN — ROUTING
# ============================================================
#
# Pattern used: ROUTING
#
# The Smart Event Planner uses a Supervisor + Workers
# multi-agent architecture.
#
# The LLM supervisor analyzes the meaning and intent of each
# user request and dynamically routes it to the appropriate
# specialist:
#
#   planning_agent
#       -> Handles event planning tasks and planning tools.
#
#   knowledge_agent
#       -> Handles knowledge-base questions using RAG.
#
# Routing decisions are made by the LLM through agent
# handoffs rather than hardcoded keyword-based conditions.
#
# Execution flow:
#
# User -> Supervisor -> Specialist Agent
#      -> Supervisor -> Final Response
#
# ============================================================

print("Workflow Pattern: ROUTING")
print("Architecture: Supervisor + Workers")
print("Routing method: LLM-based agent handoffs")

Workflow Pattern: ROUTING
Architecture: Supervisor + Workers
Routing method: LLM-based agent handoffs


In [ ]:
# ============================================================
# FINAL DEMO
# ============================================================

print("=" * 70)
print("PERSON 2 — MULTI-AGENT + ROUTING WORKFLOW DEMO")
print("=" * 70)

print("\nKnowledge Route:")
print("Supervisor -> knowledge_agent -> Supervisor")

print("\nPlanning Route:")
print("Supervisor -> planning_agent -> Supervisor")

print("\nPASS: Multi-agent routing architecture is working.")
print("PASS: Workflow Pattern used = ROUTING.")

PERSON 2 — MULTI-AGENT + ROUTING WORKFLOW DEMO

Knowledge Route:
Supervisor -> knowledge_agent -> Supervisor

Planning Route:
Supervisor -> planning_agent -> Supervisor

PASS: Multi-agent routing architecture is working.
PASS: Workflow Pattern used = ROUTING.


In [ ]:
# FINAL CAPSTONE CHECK

print("=" * 70)
print("RAG PIPELINE CAPSTONE CHECK")
print("=" * 70)

# 1. Documents
assert len(documents) > 0
print("PASS: PDF documents were loaded.")

# 2. Chunks
assert len(chunks) > 0
print(f"PASS: Documents were split into {len(chunks)} chunks.")

# 3. Embeddings
assert embedding_model is not None
print("PASS: Embedding model was created successfully.")

# 4. Vector Store
assert vector_store is not None
print("PASS: Vector store was created successfully.")

# 5. Retriever
assert retriever is not None
print("PASS: Retriever was created successfully.")

# 6. Real Retrieval
assert len(retrieved_docs) > 0
print(f"PASS: Retriever returned {len(retrieved_docs)} real document chunks.")

# 7. Sources
retrieved_sources = {
    doc.metadata.get("source", "Unknown")
    for doc in retrieved_docs
}

assert len(retrieved_sources) > 0
print("PASS: Retrieved chunks include source metadata.")

# 8. LLM Answer
assert response is not None
assert response.content.strip()
print("PASS: LLM generated an answer using retrieved context.")

# 9. Person 2 Integration
assert "search_event_knowledge" in globals()
print("PASS: RAG is integrated with Person 2 knowledge agent.")

print("\nRAG DESIGN:")
print("2-Step RAG")
print("Question -> Retrieve relevant chunks -> LLM generates grounded answer")

print("\n" + "=" * 70)
print("CAPSTONE STATUS: PASS")
print("=" * 70)

RAG PIPELINE CAPSTONE CHECK
PASS: PDF documents were loaded.
PASS: Documents were split into 3 chunks.
PASS: Embedding model was created successfully.
PASS: Vector store was created successfully.
PASS: Retriever was created successfully.
PASS: Retriever returned 3 real document chunks.
PASS: Retrieved chunks include source metadata.
PASS: LLM generated an answer using retrieved context.
PASS: RAG is integrated with Person 2 knowledge agent.

RAG DESIGN:
2-Step RAG
Question -> Retrieve relevant chunks -> LLM generates grounded answer

CAPSTONE STATUS: PASS
